<a href="https://colab.research.google.com/github/hemanya2003/Datasets/blob/main/Ganiputlanduse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install pydeck
import geopandas as gpd
import pydeck as pdk

# Load your shapefile
gdf = gpd.read_file('/content/drive/MyDrive/GANIPUT-LUSE/LUSE_FINAL.shp')

# Check the first few rows of the data
print(gdf.head())

# Check the geometry type
print("Geometry type:", gdf.geometry.type)

# Check the CRS (Coordinate Reference System)
print("Current CRS:", gdf.crs)

# Reproject to EPSG:4326 (WGS84) if necessary
if gdf.crs != 'EPSG:4326':
    print("Reprojecting to EPSG:4326...")
    gdf = gdf.to_crs(epsg=4326)

# Explode MultiPolygon geometries into individual Polygon geometries
gdf = gdf.explode(index_parts=True)

# Reset the index (optional but recommended)
gdf = gdf.reset_index(drop=True)

# Check the geometry type again
print("Geometry type after explode:", gdf.geometry.type)

# Check the bounds of the data
print("Data bounds:", gdf.total_bounds)

# Define heights for each LANDUSE category
landuse_heights = {
    'bare': 15,            # Height for 'bare'
    'built': 30,           # Height for 'built'
    'crops': 10,           # Height for 'crops'
    'grass': 20,           # Height for 'grass'
    'shrub_and_scrub': 25, # Height for 'shrub_and_scrub'
    'trees': 35,           # Height for 'trees'
    'water': -5,           # Height for 'water' (negative for depressions)
}

# Define colors for each LANDUSE category (in RGB format)
landuse_colors = {
    'bare': [255, 192, 203],       # Pink
    'built': [255, 0, 0],          # Red
    'crops': [255, 255, 0],        # Yellow
    'grass': [144, 238, 144],      # Light Green
    'shrub_and_scrub': [255, 0, 255],  # Magenta
    'trees': [0, 128, 0],          # Green
    'water': [0, 0, 255],          # Blue
}

# Assign heights and colors based on LANDUSE
gdf['height'] = gdf['LANDUSE'].map(landuse_heights)
gdf['color'] = gdf['LANDUSE'].map(landuse_colors)

# Check the updated data
print(gdf[['LANDUSE', 'height', 'color']].head())

# Replace with your MapTiler API key
MAPTILER_API_KEY = 'E6ZRsLot4eQsQImxowwm'

# Create a PyDeck layer for the polygons
polygon_layer = pdk.Layer(
    'PolygonLayer',
    data=gdf,
    get_polygon='geometry.coordinates',  # Correct property for geometry
    extruded=True,
    get_elevation='height',
    get_fill_color='color',
    elevation_scale=1,
    pickable=True,
    id='polygon-layer',
)

# Create a TileLayer for MapTiler Satellite Imagery
tile_layer = pdk.Layer(
    'TileLayer',
    data=None,
    get_tile_data=f'https://api.maptiler.com/maps/hybrid/{{z}}/{{x}}/{{y}}.jpg?key={MAPTILER_API_KEY}',
    pickable=False,
    id='tile-layer',
)

# Set the view state to center on your data
view_state = pdk.ViewState(
    latitude=gdf.geometry.centroid.y.mean(),
    longitude=gdf.geometry.centroid.x.mean(),
    zoom=10,
    pitch=45,
    bearing=30,
)

# Create the PyDeck map with both layers
r = pdk.Deck(
    layers=[tile_layer, polygon_layer],
    initial_view_state=view_state,
    map_style="https://api.maptiler.com/maps/hybrid/style.json?key=E6ZRsLot4eQsQImxowwm",
)

# Save the map as an HTML file
r.to_html('3d_landuse_map.html')

print("Map saved successfully! Open '3d_landuse_map.html' in your browser.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.7 MB/s eta 0:00:00
   OBJECTID          LANDUSE  \
0         1             bare   
1         2            built   
2         3            crops   
3         4            grass   
4         5  shrub_and_scrub   

                                            geometry  
0  MULTIPOLYGON (((82.38926 18.71495, 82.38925 18...  
1  MULTIPOLYGON (((82.3906 18.71371, 82.39039 18....  
2  MULTIPOLYGON (((82.41257 18.72365, 82.41259 18...  
3  MULTIPOLYGON (((82.39039 18.71367, 82.39031 18...  
4  MULTIPOLYGON (((82.39836 18.72924, 82.39835 18...  
Geometry type: 0    MultiPolygon
1    MultiPolygon
2    MultiPolygon
3    MultiPolygon
4    MultiPolygon
5    MultiPolygon
6    MultiPolygon
dtype: object
Current CRS: EPSG:4326
Geometry type after explode: 0      Polygon
1      Polygon
2      Polygon
3      Polygon
4      Polygon
        ...   
129    Polygon
130    Polygon
131    Polygon
132    Polygon
133    Polygon
Length: 134, dtype: object
D

<ipython-input-3-caff302e9434>:90: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  latitude=gdf.geometry.centroid.y.mean(),
<ipython-input-3-caff302e9434>:91: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  longitude=gdf.geometry.centroid.x.mean(),


<IPython.core.display.Javascript object>

Map saved successfully! Open '3d_landuse_map.html' in your browser.
